In [30]:
import pandas as pd
from datetime import datetime, timedelta
import math
import os

In [31]:
def fetch_openmeteo_archive(lat=23.5948,lon=120.442,start="2014-02-12",end="2014-04-08"):
    OPEN_METEO_API = "https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start}&end_date={end}&hourly=temperature_2m,relativehumidity_2m,precipitation,windspeed_10m,winddirection_10m&timezone=Asia%2FSingapore"
    url = OPEN_METEO_API.format(lat=lat,lon=lon,start=start,end=end)
    df = pd.read_json(url)
    df_obs = pd.DataFrame()
    for index, row in df.iterrows():
        df_obs[index] = row['hourly']
    df_obs['time'] = pd.to_datetime(df_obs['time'])
    #Calculate u, v wind components
    df_obs['u'] = df_obs['windspeed_10m'] * df_obs['winddirection_10m'].apply(lambda x: math.cos(math.radians(270-x)))
    df_obs['v'] = df_obs['windspeed_10m'] * df_obs['winddirection_10m'].apply(lambda x: math.sin(math.radians(270-x)))
    #移除有NaN的資料
    df_obs = df_obs.dropna()
    return df_obs

def fetch_openmeteo_forecast(lat=23.5948, lon=120.442, past_days=7, forecast_days=16):
    OPEN_METEO_API = "https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&hourly=temperature_2m,relativehumidity_2m,precipitation,windspeed_10m,winddirection_10m&past_days={past_days}&forecast_days={forecast_days}&models=ecmwf_aifs025&timezone=Asia%2FSingapore"
    url = OPEN_METEO_API.format(lat=lat,lon=lon,past_days=past_days,forecast_days=forecast_days)
    df = pd.read_json(url)
    df_forecast = pd.DataFrame()
    for index, row in df.iterrows():
        df_forecast[index] = row['hourly']
    df_forecast['time'] = pd.to_datetime(df_forecast['time'])
    #Calculate u, v wind components
    df_forecast['u'] = df_forecast['windspeed_10m'] * df_forecast['winddirection_10m'].apply(lambda x: math.cos(math.radians(270-x)))
    df_forecast['v'] = df_forecast['windspeed_10m'] * df_forecast['winddirection_10m'].apply(lambda x: math.sin(math.radians(270-x)))
    #移除有NaN的資料
    df_forecast = df_forecast.dropna()
    return df_forecast

In [32]:
#取得要下載的氣象站列表
STA_LIST = "https://raw.githubusercontent.com/Raingel/weather_station_list/refs/heads/main/data/weather_sta_list.csv"
df_sta = pd.read_csv(STA_LIST)
#僅保留撤站日期為nan的資料
df_sta = df_sta[df_sta['撤站日期'].isna()]
#只保留站號、站名、緯度、經度
df_sta = df_sta[['站號','站名','緯度','經度']]
#移除重複的資料
df_sta = df_sta.drop_duplicates()
print(f"共有 {len(df_sta)} 個有效氣象站")

共有 752 個有效氣象站


In [42]:
#長時間的歷史資料
past_days_start = (datetime.now() - timedelta(days=700)).strftime("%Y-%m-%d")
past_days_end = datetime.now().strftime("%Y-%m-%d")
for index, row in df_sta.iterrows():
    sta_no = row['站號']
    sta_name = row['站名']
    lat = row['緯度']
    lon = row['經度']
    #12Q970_東港工作站_22.479997_120.466058.csv
    CSV_PATH =f"../ERA5_data _20230212_20250111/{sta_no}_{sta_name}_{lat}_{lon}.csv"
    if os.path.exists(CSV_PATH):
        print(f"{sta_no} {sta_name} 的觀測資料已下載")
        continue
    print(f"開始下載 {sta_no} {sta_name} 的觀測資料")
    try:
        df_obs = fetch_openmeteo_archive(lat=lat,lon=lon,start=past_days_start,end=past_days_end)
        df_obs.to_csv(CSV_PATH, index=False)
    except Exception as e:
        print(f"下載 {sta_no} {sta_name} 的觀測資料失敗")
        print(e)

466850 五分山雷達站 的觀測資料已下載
466881 新北 的觀測資料已下載
466900 淡水 的觀測資料已下載
466910 鞍部 的觀測資料已下載
466920 臺北 的觀測資料已下載
466930 竹子湖 的觀測資料已下載
466940 基隆 的觀測資料已下載
466950 彭佳嶼 的觀測資料已下載
466990 花蓮 的觀測資料已下載
467050 新屋 的觀測資料已下載
467080 宜蘭 的觀測資料已下載
467110 金門 的觀測資料已下載
467270 田中 的觀測資料已下載
467280 後龍 的觀測資料已下載
467290 古坑 的觀測資料已下載
467300 東吉島 的觀測資料已下載
467350 澎湖 的觀測資料已下載
467410 臺南 的觀測資料已下載
467420 永康 的觀測資料已下載
467441 高雄 的觀測資料已下載
467480 嘉義 的觀測資料已下載
467490 臺中 的觀測資料已下載
467530 阿里山 的觀測資料已下載
467540 大武 的觀測資料已下載
467550 玉山 的觀測資料已下載
467571 新竹 的觀測資料已下載
467590 恆春 的觀測資料已下載
467610 成功 的觀測資料已下載
467620 蘭嶼 的觀測資料已下載
467650 日月潭 的觀測資料已下載
467660 臺東 的觀測資料已下載
467790 墾丁雷達站 的觀測資料已下載
467990 馬祖 的觀測資料已下載
C0A520 山佳 的觀測資料已下載
C0A530 坪林 的觀測資料已下載
C0A550 泰平 的觀測資料已下載
C0A570 桶後 的觀測資料已下載
C0A640 石碇 的觀測資料已下載
C0A770 科教館 的觀測資料已下載
C0A860 大坪 的觀測資料已下載
C0A870 五指山 的觀測資料已下載
C0A890 雙溪 的觀測資料已下載
C0A931 三和 的觀測資料已下載
C0A940 金山 的觀測資料已下載
C0A950 鼻頭角 的觀測資料已下載
C0A970 三貂角 的觀測資料已下載
C0A980 社子 的觀測資料已下載
C0A9C0 天母 的觀測資料已下載
C0A9F0 內湖 的觀測資料已下載
C0AC40 大屯山 的觀測資料已下載
C0AC60 三峽 的觀測資料已下載
C0AC70 信義 的觀測資

In [ ]:
lat = 23.5948
lon = 120.442
past_days_start = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")
past_days_end = datetime.now().strftime("%Y-%m-%d")
#從archive跟forecast取得氣象資料
#archive是過去的資料，forecast是未來的資料，然後將兩個資料合併，archive的優先，因為他是真實的資料
weather_df_archive = fetch_openmeteo_archive(lat=lat,lon=lon,start=past_days_start,end=past_days_end)
weather_df_forecast = fetch_openmeteo_forecast(lat=lat,lon=lon)
weather_df = pd.concat([weather_df_archive, weather_df_forecast])
weather_df = weather_df.drop_duplicates(subset=['time'])
